# 10 — Comparing the architectures and a deep ensemble

**In short:** no training here. This notebook reads the predictions saved by notebook 08
(EfficientNet-B0) and notebook 09 (the other five), scores all six the same way, applies
the winning rule, and tests whether averaging the best two or three models (a deep
ensemble, Lakshminarayanan et al., 2017) beats the best single model.

**Winning rule (fixed in #41 before the results):**
1. Highest **macro-F1 on the field validation photos** (FieldPlant + Tanzania, `val: field`).
2. Models within **0.005** of the best count as tied (about 2–3 of the 509 field validation
   photos, which is within chance); among tied models the lowest **calibration error (ECE)**
   on the same photos wins. This tolerance was written down on 14 October, before
   ConvNeXt-Tiny and Swin-Tiny had finished training.
3. The model must stay usable on the backend's CPU: at most **1,000 ms per image** here.
   The farmer already waits several seconds for the photo to upload, so up to one second
   of computing keeps the whole check within a few seconds. (An earlier draft said 200 ms;
   it was raised on 14 October, before the last two models finished, because 200 ms would
   have ruled out every ensemble by construction.)

Test, FieldPlant, Tanzania and **PlantDoc are reported for every model but never used to
choose**. PlantDoc stays the untouched "never seen before" check.

**Runtime:** CPU is enough (no GPU needed). CPU speed is measured here for all six models on
the same machine, so the times can be compared with each other.

In [ ]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis.git"
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

In [ ]:
# Keep the Drive copy of the repository in sync with GitHub.
!git -C {REPO_DIR} config core.fileMode false
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} status --short --branch

In [ ]:
%pip install -q -r {ML_DIR}/requirements.txt

## Load every model's saved predictions

Each run saved, per image, the raw scores (logits) for the 11 classes. All runs used the
same split files, so the images are in the same order in every run; this is checked,
because the ensemble averages the runs image by image.

In [ ]:
import numpy as np
import pandas as pd

from src import metrics, training

RUNS = {"efficientnet_b0": config.MODELS_DIR / "image_baseline"}
RUNS.update({arch: config.MODELS_DIR / f"compare_{arch}" for arch in training.ARCHITECTURES if arch != "efficientnet_b0"})
SETS = ["val", "test", "ood_test", "unseen_test"]
FIELD_SOURCES = ["fieldplant", "tanzania"]
CLASSES = list(config.PLANTVILLAGE_CLASSES)
INDEX = {name: i for i, name in enumerate(CLASSES)}

missing = [arch for arch, run in RUNS.items() if not (run / "results" / "ood_test_predictions.logits.npy").exists()]
assert not missing, f"Not trained yet: {missing}. Finish them in notebook 09 first."

rows, probs = {}, {arch: {} for arch in RUNS}
for arch, run in RUNS.items():
    for name in SETS:
        table = pd.read_csv(run / "results" / f"{name}_predictions.csv")
        if name in rows:
            assert table["path"].tolist() == rows[name]["path"].tolist(), f"{arch} {name}: images in a different order"
        else:
            rows[name] = table[["path", "class", "source"]]
        probs[arch][name] = metrics.softmax(np.load(run / "results" / f"{name}_predictions.logits.npy"))

print({name: len(table) for name, table in rows.items()}, "| models:", list(RUNS))

## Score every model the same way

`metrics.summarise` expects logits; the logarithm of a probability works as one (softmax
of log p gives p back), so models and ensembles go through exactly the same code.

In [ ]:
def score(p: dict) -> dict:
    """One row of the comparison table from per-set class probabilities."""
    out = {}
    def add(prefix, name, mask=None):
        labels = rows[name]["class"].map(INDEX).to_numpy()
        logits = np.log(p[name] + 1e-12)
        if mask is not None:
            labels, logits = labels[mask], logits[mask]
        s = metrics.summarise(logits, labels, len(CLASSES))
        for key in ["accuracy", "macro_f1", "ece", "mean_confidence"]:
            out[f"{prefix} {key}"] = s[key]

    val_sources = rows["val"]["source"].to_numpy()
    add("val field", "val", np.isin(val_sources, FIELD_SOURCES))
    add("test", "test")
    test_sources = rows["test"]["source"].to_numpy()
    for source in ["fieldplant", "tanzania"]:
        add(f"test {source}", "test", test_sources == source)
    add("plantdoc", "ood_test")
    unseen = p["unseen_test"].max(axis=1)
    out["unseen mean_confidence"] = unseen.mean()
    out["unseen share >= 0.90"] = (unseen >= 0.9).mean()
    return out

table = pd.DataFrame({arch: score(probs[arch]) for arch in RUNS}).T
display(table[["val field macro_f1", "val field ece", "test accuracy", "test fieldplant accuracy",
               "test tanzania accuracy", "plantdoc accuracy", "plantdoc mean_confidence"]].round(4))

## How sure are those field validation scores?

The field validation set has 509 photos, so small differences can be luck. A bootstrap
redraws the 509 photos with replacement 1,000 times and recomputes macro-F1 each time;
the middle 95% of those values is the uncertainty range.

In [ ]:
from sklearn.metrics import f1_score

field_mask = np.isin(rows["val"]["source"].to_numpy(), FIELD_SOURCES)
field_labels = rows["val"]["class"].map(INDEX).to_numpy()[field_mask]
rng = np.random.default_rng(config.RANDOM_SEED)
draws = [rng.integers(0, len(field_labels), len(field_labels)) for _ in range(1000)]

def interval(p_val):
    predicted = p_val[field_mask].argmax(axis=1)
    values = [f1_score(field_labels[d], predicted[d], average="macro", labels=sorted(set(field_labels[d])), zero_division=0)
              for d in draws]
    return np.percentile(values, [2.5, 97.5])

for arch in RUNS:
    low, high = interval(probs[arch]["val"])
    table.loc[arch, "val field macro_f1 95% low"], table.loc[arch, "val field macro_f1 95% high"] = low, high
display(table[["val field macro_f1", "val field macro_f1 95% low", "val field macro_f1 95% high"]].round(4))

## Size and CPU speed

Parameters and file size say whether a model could later run on a phone; CPU time per
image is what the backend needs. All six are timed here, on this one machine, with 2
threads and a single 224×224 image.

In [ ]:
import torch

for arch, run in RUNS.items():
    model = training.build_model(len(CLASSES), arch, pretrained=False)
    table.loc[arch, "parameters (M)"] = training.count_parameters(model) / 1e6
    table.loc[arch, "file (MB)"] = (run / "model.pt").stat().st_size / 1e6
    table.loc[arch, "cpu ms/image"] = training.cpu_milliseconds_per_image(model, 224)
    del model
display(table[["parameters (M)", "file (MB)", "cpu ms/image"]].round(1))

## Apply the winning rule

In [ ]:
TIE = 0.005
MAX_CPU_MS = 1000

usable = table[table["cpu ms/image"] <= MAX_CPU_MS]
best_f1 = usable["val field macro_f1"].max()
tied = usable[usable["val field macro_f1"] >= best_f1 - TIE]
winner = tied["val field ece"].idxmin()
ranking = table.sort_values(["val field macro_f1", "val field ece"], ascending=[False, True]).index.tolist()

print("Too slow for the backend:", sorted(set(table.index) - set(usable.index)) or "none")
print(f"Best field validation macro-F1: {best_f1:.4f}; tied within {TIE}: {tied.index.tolist()}")
print("Best single model:", winner)

## Deep ensemble of the best models

Each photo's class probabilities are averaged over the members. Members are taken in the
order of the ranking above (field validation macro-F1), so the choice of members also uses
validation data only. An ensemble costs the sum of its members' CPU time.

In [ ]:
candidates = {}
for k in [2, 3]:
    members = ranking[:k]
    name = "ensemble: " + " + ".join(members)
    averaged = {s: np.mean([probs[a][s] for a in members], axis=0) for s in SETS}
    probs[name] = averaged
    candidates[name] = members
    scores = score(averaged)
    table.loc[name, list(scores)] = list(scores.values())
    low, high = interval(averaged["val"])
    table.loc[name, ["val field macro_f1 95% low", "val field macro_f1 95% high"]] = [low, high]
    table.loc[name, "cpu ms/image"] = table.loc[members, "cpu ms/image"].sum()
    table.loc[name, "parameters (M)"] = table.loc[members, "parameters (M)"].sum()
    table.loc[name, "file (MB)"] = table.loc[members, "file (MB)"].sum()

display(table.loc[[winner, *candidates], ["val field macro_f1", "val field ece", "plantdoc accuracy",
                                           "plantdoc ece", "plantdoc mean_confidence", "cpu ms/image"]].round(4))

## The final choice

The same rule decides between the best single model and the ensembles.

In [ ]:
final_pool = table.loc[[winner, *candidates]]
final_pool = final_pool[final_pool["cpu ms/image"] <= MAX_CPU_MS]
top = final_pool["val field macro_f1"].max()
final_tied = final_pool[final_pool["val field macro_f1"] >= top - TIE]
choice = final_tied["val field ece"].idxmin()
print("Chosen for the backend:", choice)
print(table.loc[choice, ["val field macro_f1", "val field ece", "test accuracy", "plantdoc accuracy",
                         "plantdoc mean_confidence", "cpu ms/image"]].round(4).to_string())

## The overconfidence on PlantDoc, model by model

For every model the gap between how often it is right on PlantDoc and how sure it says
it is. This gap is what calibration and the referral threshold (Sprint 4) deal with.

In [ ]:
import matplotlib.pyplot as plt

ACCURACY_COLOUR, CONFIDENCE_COLOUR = "#2a78d6", "#eb6834"  # categorical slots 1 and 2
order = table.sort_values("plantdoc accuracy").index.tolist()
y = np.arange(len(order))
accuracy = table.loc[order, "plantdoc accuracy"].to_numpy() * 100
confidence = table.loc[order, "plantdoc mean_confidence"].to_numpy() * 100

fig, ax = plt.subplots(figsize=(9, 0.55 * len(order) + 1.4))
ax.hlines(y, accuracy, confidence, color="#c3c2b7", linewidth=2, zorder=1)
ax.scatter(accuracy, y, s=70, color=ACCURACY_COLOUR, edgecolor="white", linewidth=2, zorder=2, label="accuracy (how often right)")
ax.scatter(confidence, y, s=70, color=CONFIDENCE_COLOUR, edgecolor="white", linewidth=2, zorder=2, label="mean confidence (how sure it says it is)")
for i in range(len(order)):
    ax.text(accuracy[i] - 1.5, y[i], f"{accuracy[i]:.0f}%", va="center", ha="right", fontsize=8, color="#444441")
    ax.text(confidence[i] + 1.5, y[i], f"{confidence[i]:.0f}%", va="center", ha="left", fontsize=8, color="#444441")
ax.set_yticks(y, [name.replace("ensemble: ", "ensemble:\n") for name in order], fontsize=8)
ax.set_xlim(0, 105)
ax.set_xlabel("PlantDoc photos (%)")
ax.set_title("Every model is far more confident than it is right on unseen field photos", fontsize=10, loc="left")
ax.grid(axis="x", color="#e6e5e0", linewidth=0.8)
ax.set_axisbelow(True)
for side in ["top", "right", "left"]:
    ax.spines[side].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.06), ncol=2, frameon=False, fontsize=8)
plt.tight_layout()

OUT = ML_DIR / "results" / "comparison"
OUT.mkdir(parents=True, exist_ok=True)
plt.savefig(OUT / "plantdoc_overconfidence.png", dpi=150)
plt.show()

## Save the comparison

The full table and the decision go to `ml/results/comparison/` so they are visible on GitHub.

In [ ]:
import json

table.round(4).to_csv(OUT / "comparison.csv")
decision = {
    "rule": f"highest val field macro-F1; within {TIE} counts as tied, then lowest val field ECE; at most {MAX_CPU_MS} ms per image on CPU",
    "ranking_by_val_field_macro_f1": ranking,
    "best_single_model": winner,
    "ensembles": candidates,
    "chosen": choice,
}
(OUT / "decision.json").write_text(json.dumps(decision, indent=2))
print(json.dumps(decision, indent=2))

## Make sure Drive has saved everything, then save to GitHub

Save the notebook first (Ctrl+S). Only `ml/results/comparison/` is committed, with a copy of
this notebook and its outputs.

In [ ]:
drive.flush_and_unmount()
drive.mount("/content/drive")
print("Drive synced and remounted.")

In [ ]:
import shutil

from google.colab import userdata

from src import github_sync

shutil.copy(ML_DIR / "notebooks" / "10_architecture_comparison_results.ipynb", OUT / "notebook.ipynb")
print(github_sync.push(
    REPO_DIR,
    branch="colab/architecture-comparison-results",
    message="Compare the six architectures and a deep ensemble",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
    paths=[str(OUT.relative_to(REPO_DIR))],
))